# Agent Evaluation — Exact Match and Semantic Scoring
## Did It Get the Right Answer? — Exact Match and Semantic Scoring
⏱ ~12 min

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/Esturban/agent/blob/master/examples/76-agent-evaluation/agent_evaluation_workbook.ipynb)

Evaluates an LLM agent against a 5-question golden QA set using two metrics: exact match (keyword in answer) and semantic match (cosine similarity ≥ 0.80 between expected and actual embeddings).

---

### Workshop Roadmap

| # | Topic |
|---|-------|
| 1 | **Concepts** — Why automated evaluation matters; exact vs semantic match |
| 2 | **Golden Dataset** — 5 questions with expected answers |
| 3 | **run_agent()** — LLM answers each question; compare to expected |
| 4 | **evaluate_answer()** — Exact match + cosine similarity via embeddings |
| 5 | **Full Eval Loop** — Score all 5, aggregate pass rate |
| ★ | **Exercises + Answer Key** |

### Prerequisites
- Python 3.10+, or Google Colab (install cell below)
- `OPENAI_API_KEY` in `.env` or Colab Secrets

In [ ]:
import sys

def _in_colab():
    try:
        import google.colab; return True
    except ImportError:
        return False

if _in_colab():
    import subprocess
    subprocess.run([sys.executable, "-m", "pip", "install", "-q",
        "langchain", "langchain-openai", "numpy", "langgraph", "python-dotenv"])
    print("Colab install complete.")
else:
    print("Local — skipping install.")

In [ ]:
import os
try:
    from google.colab import userdata
    os.environ["OPENAI_API_KEY"] = userdata.get("OPENAI_API_KEY")
except ImportError:
    from dotenv import load_dotenv; load_dotenv()

key = os.environ.get("OPENAI_API_KEY", "")
print(f"API key ready: {bool(key) and key.startswith('sk-')}")

In [ ]:
import numpy as np
from typing import TypedDict
from langchain_core.messages import HumanMessage
from langchain_openai import ChatOpenAI, OpenAIEmbeddings
from langgraph.graph import END, START, StateGraph

# golden set: hand-curated question/answer pairs that act as the ground truth for regression testing
GOLDEN_QA_SET = [
    {"question": "What is the capital of France?",      "expected": "Paris"},
    {"question": "What is 2 + 2?",                      "expected": "4"},
    {"question": "Who wrote Romeo and Juliet?",          "expected": "Shakespeare"},
    {"question": "What is the boiling point of water in Celsius?", "expected": "100"},
    {"question": "What planet is closest to the Sun?",  "expected": "Mercury"},
]

# temperature=0 for deterministic answers — makes eval reproducible; raise it only for creative tasks
agent_llm   = ChatOpenAI(model="gpt-4o-mini", temperature=0)
embed_model = OpenAIEmbeddings(model="text-embedding-3-small")

def run_agent(question: str) -> str:
    return agent_llm.invoke([HumanMessage(content=f"Answer briefly in one sentence: {question}")]).content.strip()

def evaluate_answer(expected: str, actual: str) -> dict:
    exact = expected.lower() in actual.lower()
# embed both strings independently — cosine similarity captures meaning even when phrasing differs
    ev = np.array(embed_model.embed_query(expected))
    av = np.array(embed_model.embed_query(actual))
    sim = float(np.dot(ev, av) / (np.linalg.norm(ev) * np.linalg.norm(av)))
# 0.80 cosine threshold is a sensible default — lower it to ~0.70 for paraphrase-heavy domains
    semantic = sim >= 0.80
    return {"exact_match": exact, "semantic_match": semantic,
            "cosine_similarity": round(sim, 4),
            "score": 1.0 if (exact or semantic) else round(sim, 4)}

# TypedDict gives LangGraph typed access to state keys without runtime overhead vs dataclasses
class EvalState(TypedDict):
    question: str; expected: str; actual: str; result: dict

# evaluate_one is the single graph node — keeping it atomic makes the graph easy to extend with retry logic
def evaluate_one(state):
    actual = run_agent(state["question"])
    result = evaluate_answer(state["expected"], actual)
    return {**state, "actual": actual, "result": result}

# StateGraph routes messages through typed state — EvalState keys are the contract between nodes
g = StateGraph(EvalState)
g.add_node("evaluate_one", evaluate_one)
g.add_edge(START, "evaluate_one"); g.add_edge("evaluate_one", END)
# compile() locks the graph topology — required before invoke; add MemorySaver() here for cross-run persistence
app = g.compile()

# invoke once per row — straightforward for small golden sets; for 100+ rows, batch with async invoke
results = []
for row in GOLDEN_QA_SET:
    r = app.invoke({"question": row["question"], "expected": row["expected"], "actual": "", "result": {}})
    results.append(r)
    icon = "✓" if r["result"]["score"] == 1.0 else "~"
    print(f"{icon} Q: {r['question']}")
    print(f"  Expected: {r['expected']}  |  Got: {r['actual'][:60]}")
    print(f"  exact={r['result']['exact_match']}  semantic={r['result']['semantic_match']}  sim={r['result']['cosine_similarity']}")

# pass_rate counts exact-or-semantic wins; avg_sim is a softer signal for near-misses
pass_rate = sum(1 for r in results if r["result"]["score"] == 1.0) / len(results)
avg_sim   = sum(r["result"]["cosine_similarity"] for r in results) / len(results)
print(f"\nPass rate: {pass_rate:.0%}  |  Avg cosine: {avg_sim:.3f}")

---
*Workshop complete. Pausing — confirm to continue to the next notebook.*